> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/DataProcessing/blob/main/notebooks/01b_Cleaning_Points_and_Polygons.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Cleaning Points and Polygons

**Series:** Cleaning and Preparing Data in Python (notebook 2 of 2)
**Time:** about 35 minutes

**In this notebook you will:**
- Turn a raw table with coordinates into points with the right CRS
- Recode numeric codes into readable labels and clean up text fields
- Tell duplicate records from different records that share a location
- Check geometries, remove features you do not want and merge split features
- Check a cleaned layer against the source with a spatial join

**You should already know:** notebook 01a.

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## Two raw layers

| File | What it is | Source |
|---|---|---|
| `ipeds_hd2023_indiana_raw.csv` | Every Indiana college in the federal IPEDS directory (2023), with coordinates; 19 of its 73 columns | U.S. Department of Education, NCES |
| `osm_parks_tippecanoe_raw.geojson` | Every OpenStreetMap feature tagged `leisure=park` in Tippecanoe County, one per OSM object | OpenStreetMap contributors (Overpass API) |

Both are the raw downloads: nothing was fixed. The Foundations course uses cleaned versions of both (70 colleges,
68 parks); here you make those clean versions yourself.

## Part A: colleges

### Step 1: Inspect

In [ ]:
import pandas as pd
import geopandas as gpd

DP = "https://raw.githubusercontent.com/SpatialTurn/DataProcessing/main/data"
raw = pd.read_csv(f"{DP}/ipeds_hd2023_indiana_raw.csv")
print(raw.shape)
raw.head(3)

In [ ]:
raw[["ZIP", "SECTOR", "DEGGRANT", "C21BASIC", "CLOSEDAT"]].describe(include="all").T

What to notice:

- **ZIP codes come in two forms**, `47906` and `47907-2040` (ZIP+4). Here pandas read them as text because of the
  dash; in a file with only 5-digit ZIPs it would read numbers and drop the leading zeros of ZIPs such as `02138`.
- **Coded columns.** `SECTOR`, `CONTROL`, `ICLEVEL` and `DEGGRANT` are numbers that stand for categories
  (`SECTOR` 1 = public, 4-year or above). The codebook (the data dictionary) says what each code means.
- **Negative codes.** IPEDS uses `-2` for "not applicable" and `-1` for "not reported". `CLOSEDAT` is `-2` in
  every row (none of these colleges has closed), and `C21BASIC` (the Carnegie classification) is `-2` for many.

### Step 2: Read codes as text and turn "not applicable" into missing

In [ ]:
raw = pd.read_csv(f"{DP}/ipeds_hd2023_indiana_raw.csv",
                  dtype={"UNITID": str, "ZIP": str, "COUNTYCD": str})
col = raw.copy()
col["ZIP5"] = col["ZIP"].str[:5]
for c in ["C21BASIC", "CLOSEDAT", "INSTSIZE"]:
    col[c] = col[c].mask(col[c] < 0)
print(col[["C21BASIC", "CLOSEDAT", "INSTSIZE"]].isna().sum())

### Step 3: Recode categories into labels

Write the codebook as a dictionary and `map` it. Any code missing from the dictionary becomes `NaN`, so count
those too: a missing label means the dictionary is incomplete.

In [ ]:
SECTOR = {0: "Administrative unit", 1: "Public, 4-year or above", 2: "Private nonprofit, 4-year or above",
          3: "Private for-profit, 4-year or above", 4: "Public, 2-year", 5: "Private nonprofit, 2-year",
          6: "Private for-profit, 2-year", 7: "Public, less than 2-year",
          8: "Private nonprofit, less than 2-year", 9: "Private for-profit, less than 2-year"}
col["sector"] = col["SECTOR"].map(SECTOR)
print("unlabeled:", col["sector"].isna().sum())
col["sector"].value_counts()

### Step 4: Keep the rows you need

For a study of colleges that award degrees, keep `DEGGRANT == 1` (degree-granting). The others are mostly
private for-profit schools with programs shorter than two years. Write the rule down: it is a decision, not a fix.

In [ ]:
colleges = col[col["DEGGRANT"] == 1].copy()
print(len(col), "institutions,", len(colleges), "degree-granting")

### Step 5: Duplicates, or not?

Check the identifier first: every `UNITID` should appear once. Then look for rows that share a **location**.

In [ ]:
print("duplicate UNITIDs:", colleges["UNITID"].duplicated().sum())
same_place = colleges[colleges.duplicated(["LONGITUD", "LATITUDE"], keep=False)]
same_place[["UNITID", "INSTNM", "CITY", "LONGITUD", "LATITUDE"]].sort_values("LONGITUD")

Two pairs of colleges share exact coordinates: Trine University and its regional campus, and Indiana Tech and
its College of Professional Studies. They are **different institutions** reported at the same address, not
duplicates. A tool that deletes duplicate geometries (QGIS has one) would silently remove two real colleges.
Before you delete a "duplicate", decide what makes two records the same: here it is the `UNITID`.

### Step 6: Make points, with the right CRS

IPEDS gives longitude and latitude in decimal degrees, so the CRS is geographic (EPSG:4326). Longitude is x,
latitude is y; swapping them puts Indiana in Antarctica.

In [ ]:
colleges = gpd.GeoDataFrame(colleges, geometry=gpd.points_from_xy(colleges["LONGITUD"], colleges["LATITUDE"]),
                            crs="EPSG:4326")
print(colleges.total_bounds.round(2))        # x from -88 to -84.8, y from 37.8 to 41.8 is Indiana

### Step 7: Check against the map

Does each college fall inside the county IPEDS says it is in? A spatial join answers that for all 70 at once.

In [ ]:
counties = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/indiana_counties.geojson")[["name", "fips", "geometry"]]
joined = gpd.sjoin(colleges, counties, how="left", predicate="within")
print("outside every county:", joined["index_right"].isna().sum())
mismatch = joined[joined["COUNTYCD"] != joined["fips"].astype(str)]
print("county code disagrees with location:", len(mismatch))

Both checks pass: every point lies in Indiana, in the county its record names. A check that finds nothing is
still worth running; now you *know* the coordinates are right instead of hoping so. Finally, keep readable
columns:

In [ ]:
colleges = colleges.rename(columns={"UNITID": "unitid", "INSTNM": "name", "CITY": "city", "COUNTYNM": "county"})
colleges = colleges[["unitid", "name", "city", "county", "ZIP5", "sector", "geometry"]]
colleges.head(3)

## Part B: parks

### Step 1: Inspect

In [ ]:
parks_raw = gpd.read_file(f"{DP}/osm_parks_tippecanoe_raw.geojson")
print(len(parks_raw), "features,", parks_raw.crs)
print("without a name:", parks_raw["name"].isna().sum())
print("names used more than once:", parks_raw["name"].value_counts()[lambda s: s > 1].to_dict())

OpenStreetMap is edited by volunteers, and three things are typical:

- **Unnamed features** (10 here): small green areas, often pieces of a larger park or private lawns.
- **One park, several features**: McAllister Park and Triangle Park are each mapped as two touching shapes.
- **Geometry errors**: a polygon whose edges cross itself is *invalid* and makes overlays fail or give wrong
  areas.

### Step 2: Check the geometries

In [ ]:
print("invalid:", (~parks_raw.is_valid).sum(), " empty:", parks_raw.is_empty.sum(),
      " missing:", parks_raw.geometry.isna().sum())

All valid. That will not always be so; this is what an invalid polygon looks like, and how `make_valid` repairs
it:

In [ ]:
from shapely.geometry import Polygon
from shapely import make_valid

bowtie = Polygon([(0, 0), (2, 2), (2, 0), (0, 2)])      # edges cross in the middle
print(bowtie.is_valid, bowtie.area)                      # invalid, and its area is wrong (0)
fixed = make_valid(bowtie)
print(fixed.geom_type, fixed.is_valid, fixed.area)       # two triangles, area 2

In GeoPandas, `gdf.geometry = gdf.make_valid()` repairs a whole layer. Check first; repair only what is
broken, and look at the repaired features on a map.

### Step 3: Remove what you do not want, merge what belongs together

In [ ]:
named = parks_raw[parks_raw["name"].notna()]
parks = named.dissolve(by="name", as_index=False, aggfunc={"osm_id": lambda s: ";".join(s)})
print(len(parks_raw), "->", len(named), "named ->", len(parks), "parks")

### Step 4: Reproject and measure

Areas need a CRS in meters (UTM zone 16N, EPSG:26916).

In [ ]:
parks = parks.to_crs(epsg=26916)
parks["area_ha"] = (parks.area / 10_000).round(2)
print(parks.sort_values("area_ha", ascending=False)[["name", "area_ha"]].head(5).to_string(index=False))

clean = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_parks.geojson")
print("same names as the Foundations parks file:", set(parks["name"]) == set(clean["name"]))

In [ ]:
ax = parks.plot(color="#31a354", figsize=(7, 7))
parks_raw[parks_raw["name"].isna()].to_crs(epsg=26916).plot(ax=ax, color="#e6550d")
ax.set_title("Named parks (green) and unnamed features removed (orange)")
ax.set_axis_off()

## Try it yourself

1. How many degree-granting colleges does each `sector` have?
2. Which Indiana county has the most degree-granting colleges?

*Hint:* `value_counts()` on a column counts each value.

<details>
<summary><b>Show a solution</b></summary>

```python
print(colleges["sector"].value_counts())
print(colleges["county"].value_counts().head(3))
```

</details>

In [ ]:
# Write your answer here

## Recap

- Points from coordinates: longitude is x, latitude is y, and decimal degrees mean EPSG:4326
- Read codes and identifiers as text; turn negative 'not applicable' codes into missing values
- Recode categories with a codebook dictionary, and count the codes it does not cover
- Decide what makes two records the same (an ID), before removing duplicates
- Check validity, remove unwanted features, dissolve split features, reproject, and compare with the source

**Next:** the Geoprocessing module, which uses the cleaned tracts, parks and stores to ask what lies within a
given distance.